In [1]:
!pip install axelrod

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 33.0 MB/s eta 0:00:00:00:010:01m


In [5]:
"""
Approach A v3-fixed
====================
Identical to v3 except ONE bug fixed in extract_features:

  BUG (v3):
      dstreak = sum(1 for _ in iter(lambda: oh and oh.pop() == D ..., False))
      dstreak = 0   # overwrote the above anyway — but the .pop() already
                    # emptied `oh` in place, corrupting every feature after it

  FIX: simple reversed-loop that does not mutate oh

  Effect of bug:
    - oh was emptied mid-feature-extraction
    - All downstream features (early_dr, late_dr, perm_ret, alt_score, …)
      received oh=[], producing near-constant 0/0.5 values
    - Zero-variance features → LayerNorm gradient explosion → NaN loss
    - Classifier trained to ~27% accuracy (chance) → agent effectively random
    - TFT-like and Grudger opponents scored 0.367 (same as AllD)

  Everything else — architecture, training loop, action selection,
  cooperative prior, specialist vote, defector override — is unchanged.
"""

import os, random, time, json, warnings
from typing import List, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")

try:
    import axelrod as axl
except ImportError:
    print("[ERROR] pip install axelrod"); raise

try:
    from tqdm import tqdm
    _HAS_TQDM = True
except ImportError:
    tqdm = lambda x, **kw: x
    _HAS_TQDM = False

# ─────────────────────────────────────────────────────────────────────────────
SEED   = 42
C, D   = 0, 1
ROUNDS = 100
PAYOFF = {(C,C):3, (C,D):0, (D,C):5, (D,D):1}

TYPE_TFT      = 0
TYPE_GRUDGER  = 1
TYPE_DEFECTOR = 2
TYPE_STOCH    = 3
TYPE_COOP     = 4
TYPE_CYCLER   = 5
NUM_TYPES     = 6

FAMILY_TO_TYPE = {
    "TFT-Like":       TYPE_TFT,
    "Grudger-Like":   TYPE_GRUDGER,
    "Defector-Like":  TYPE_DEFECTOR,
    "Stochastic":     TYPE_STOCH,
    "Cooperative":    TYPE_COOP,
    "PhaseSwitching": TYPE_CYCLER,
    "Adaptive":       TYPE_TFT,
    "Sinusoidal":     TYPE_STOCH,
}

# Cooperative prior: bias toward assuming cooperative opponent until proven otherwise
PRIOR = np.array([0.30, 0.25, 0.05, 0.15, 0.15, 0.10], dtype=np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seeds()
print(f"[A-v3-fixed] Device: {device}")

def a2i(a): return 0 if a == axl.Action.C else 1
def i2a(i): return axl.Action.C if i == 0 else axl.Action.D

# ─────────────────────────────────────────────────────────────────────────────
# FEATURES  (20-dim)
# ─────────────────────────────────────────────────────────────────────────────
def extract_features(ph: List[int], oh: List[int], rnd: int) -> np.ndarray:
    """
    Extract 20 diagnostic features from interaction history.
    NOTE: does NOT mutate ph or oh.
    """
    n = max(1, rnd)
    oh_arr = np.array(oh, dtype=np.float32) if oh else np.zeros(1)
    ph_arr = np.array(ph, dtype=np.float32) if ph else np.zeros(1)

    opp_dr = float(oh_arr.mean())
    my_dr  = float(ph_arr.mean())

    r0_def = float(len(oh) > 0 and oh[0] == D)
    r1_def = float(len(oh) > 1 and oh[1] == D)
    r2_def = float(len(oh) > 2 and oh[2] == D)

    def cond_rate(my_act, next_opp):
        pairs = [(ph[i], oh[i+1]) for i in range(len(ph)-1)
                 if ph[i] == my_act and i+1 < len(oh)]
        if not pairs: return 0.5
        return float(np.mean([b == next_opp for _, b in pairs]))

    coop_after_c = cond_rate(C, C)
    def_after_c  = cond_rate(C, D)
    coop_after_d = cond_rate(D, C)
    def_after_d  = cond_rate(D, D)

    mirror = 0.5
    if len(ph) >= 2:
        mirror = float(np.mean([oh[t] == ph[t-1]
                                 for t in range(1, min(len(oh), len(ph)))]))

    ever_def = any(a == D for a in oh)
    perm_ret = 0.0
    if ever_def:
        first_d = next(i for i, a in enumerate(oh) if a == D)
        perm_ret = float(all(a == D for a in oh[first_d:]))

    alt_score = 0.0
    if len(oh) >= 4:
        alt_score = float(np.mean([oh[i] != oh[i+1] for i in range(len(oh)-1)]))

    opp_var = float(np.var(1 - oh_arr)) if len(oh_arr) >= 3 else 0.25

    # FIXED: simple loop — does not mutate oh
    dstreak = 0
    for a in reversed(oh):
        if a == D: dstreak += 1
        else: break
    dstreak_f = min(dstreak / 10.0, 1.0)

    early_dr    = float(np.mean(oh[:20])) if oh else 0.5
    late_dr     = float(np.mean(oh[20:])) if len(oh) > 20 else (float(np.mean(oh)) if oh else 0.5)
    mc          = float(sum(1 for a,b in zip(ph,oh) if a==C and b==C) / n)
    exploit     = float(sum(1 for a,b in zip(ph,oh) if a==D and b==C) / n)
    progress    = rnd / ROUNDS
    reliability = min(rnd / 20.0, 1.0)

    return np.array([
        opp_dr, my_dr, r0_def, r1_def, r2_def,
        coop_after_c, def_after_c, coop_after_d, def_after_d,
        mirror, perm_ret, alt_score, opp_var,
        dstreak_f, early_dr, late_dr, mc, exploit,
        progress, reliability,
    ], dtype=np.float32)

FEAT_DIM = 20

# ─────────────────────────────────────────────────────────────────────────────
# SPECIALIST POLICIES
# ─────────────────────────────────────────────────────────────────────────────
def policy_allc(ph, oh, r):  return C
def policy_alld(ph, oh, r):  return D
def policy_tft(ph, oh, r):   return C if not oh else oh[-1]
def policy_grim(ph, oh, r):  return D if any(a==D for a in oh) else C

def policy_gtft(ph, oh, r, p=0.15):
    if not oh: return C
    return (D if random.random() > p else C) if oh[-1] == D else C

def policy_cycler(ph, oh, r):
    """Detect opponent period and exploit cooperative phases."""
    for period in [2, 3, 4, 5, 6]:
        if len(oh) >= 2 * period:
            if oh[-2*period:-period] == oh[-period:]:
                pred_next = oh[-period]
                return D if pred_next == C else C
    return policy_tft(ph, oh, r)

# Indexed by TYPE_*
SPECIALISTS = [
    policy_tft,     # TYPE_TFT
    policy_allc,    # TYPE_GRUDGER  — never trigger permanent retaliation
    policy_alld,    # TYPE_DEFECTOR
    policy_gtft,    # TYPE_STOCH
    policy_allc,    # TYPE_COOP
    policy_cycler,  # TYPE_CYCLER
]

# ─────────────────────────────────────────────────────────────────────────────
# CLASSIFIER
# ─────────────────────────────────────────────────────────────────────────────
class OpponentClassifier(nn.Module):
    def __init__(self, in_dim=FEAT_DIM, num_types=NUM_TYPES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),  nn.LayerNorm(64),  nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_types)
        )
    def forward(self, x):    return self.net(x)
    def probs(self, x):      return F.softmax(self.forward(x), dim=-1)
    def predict(self, x):    return int(self.probs(x).argmax(-1).item())
    def confidence(self, x): return float(self.probs(x).max(-1).values.item())

# ─────────────────────────────────────────────────────────────────────────────
# ENVIRONMENT
# ─────────────────────────────────────────────────────────────────────────────
class IPDEnv:
    def __init__(self, opp_class, opp_kwargs=None, family="unknown"):
        self.opp_class  = opp_class
        self.opp_kwargs = opp_kwargs or {}
        self.family     = family
        self.reset()

    def reset(self):
        self.opp = self.opp_class(**self.opp_kwargs)
        self.opp.reset()
        self.ph, self.oh = [], []
        self.rnd = 0
        return extract_features([], [], 0)

    def step(self, action):
        proxy_history = [i2a(a) for a in self.ph]
        class Proxy:
            history = proxy_history
            match_winner = None
            match_attributes = {}
        try:
            opp_act = a2i(self.opp.strategy(Proxy()))
        except Exception:
            opp_act = self.oh[-1] if self.oh else C
        rew = float(PAYOFF[(action, opp_act)])
        self.ph.append(action); self.oh.append(opp_act)
        if hasattr(self.opp, 'history'):
            try: self.opp.history.append(i2a(opp_act), i2a(action))
            except: pass
        self.rnd += 1
        done = self.rnd >= ROUNDS
        return extract_features(self.ph, self.oh, self.rnd), rew, done, {"opp_action": opp_act}

# ─────────────────────────────────────────────────────────────────────────────
# META-AGENT
# ─────────────────────────────────────────────────────────────────────────────
class MetaAgent:
    """
    Action selection:
      - Rounds 0-1: always cooperate (warm start)
      - Round 2+:   update classifier every 3 rounds
      - Defector override: only if WE cooperated and THEY defected 4+ times
      - Action = weighted specialist vote (sum p[t] * I(spec_t == C) vs D)
    """
    name = "MetaAgent-A-v3-fixed"

    def __init__(self, lr_cls=3e-4):
        self.classifier = OpponentClassifier().to(device)
        self.cls_opt    = optim.Adam(self.classifier.parameters(), lr=lr_cls,
                                     weight_decay=1e-4)
        self._reset_ep()

    def _reset_ep(self):
        self._ph: List[int] = []
        self._oh: List[int] = []
        self._rnd = 0
        self._type_probs = PRIOR.copy()

    def on_episode_start(self):  self._reset_ep()
    def on_episode_end(self):    pass

    def on_step(self, action, opp_action, **kw):
        self._ph.append(action)
        self._oh.append(opp_action)
        self._rnd += 1

    def _update_probs(self):
        feat = torch.tensor(
            extract_features(self._ph, self._oh, self._rnd),
            dtype=torch.float32
        ).unsqueeze(0).to(device)
        with torch.no_grad():
            classifier_probs = self.classifier.probs(feat).cpu().numpy()[0]
        reliability = min(self._rnd / 15.0, 1.0)
        self._type_probs = (reliability * classifier_probs +
                            (1 - reliability) * PRIOR)

    def _is_unconditional_defector(self) -> bool:
        """True only if opponent defected 4+ times while WE cooperated."""
        if self._rnd < 4:
            return False
        my_coop_rounds = [i for i, a in enumerate(self._ph) if a == C]
        if len(my_coop_rounds) < 4:
            return False
        opp_defs_when_we_cooped = sum(
            1 for i in my_coop_rounds if i < len(self._oh) and self._oh[i] == D
        )
        return opp_defs_when_we_cooped >= 4

    def select_action(self, obs, eval_mode=False) -> int:
        rnd = self._rnd
        ph, oh = self._ph, self._oh

        # Warm start
        if rnd < 2:
            return C

        # Update classifier every 3 rounds
        if rnd % 3 == 0:
            self._update_probs()

        # Defector override (requires we were cooperating — prevents self-spirals)
        if self._is_unconditional_defector():
            return D

        p = self._type_probs

        # Weighted specialist vote
        vote_c = sum(p[t] for t in range(NUM_TYPES) if SPECIALISTS[t](ph, oh, rnd) == C)
        vote_d = sum(p[t] for t in range(NUM_TYPES) if SPECIALISTS[t](ph, oh, rnd) == D)

        return C if vote_c >= vote_d else D

    def train_classifier(self, episodes: List[Dict], n_epochs=30) -> float:
        if not episodes: return 0.0
        features = torch.tensor(
            np.array([e["features"] for e in episodes]),
            dtype=torch.float32
        ).to(device)
        labels = torch.tensor(
            [e["true_type"] for e in episodes],
            dtype=torch.long
        ).to(device)

        counts  = torch.bincount(labels, minlength=NUM_TYPES).float()
        weights = (1.0 / counts.clamp(min=1)).to(device)
        weights /= weights.sum()

        losses = []
        for _ in range(n_epochs):
            logits = self.classifier(features)
            loss   = F.cross_entropy(logits, labels, weight=weights)
            self.cls_opt.zero_grad()
            loss.backward()
            self.cls_opt.step()
            losses.append(float(loss.item()))
        return float(np.mean(losses))

# ─────────────────────────────────────────────────────────────────────────────
# DATA COLLECTION  (agent plays its own policy — no distribution mismatch)
# ─────────────────────────────────────────────────────────────────────────────
def collect_episodes(pool, agent, n_eps_per_opp=30, seed=SEED) -> List[Dict]:
    cls_data = []
    for entry in pool:
        true_type = FAMILY_TO_TYPE.get(entry["family"], TYPE_STOCH)
        for ep_idx in range(n_eps_per_opp):
            env = IPDEnv(entry["class"], entry["kwargs"], entry["family"])
            env.reset()
            agent.on_episode_start()
            ph, oh = [], []

            for rnd in range(ROUNDS):
                obs = extract_features(ph, oh, rnd)
                act = agent.select_action(obs)
                _, rew, done, info = env.step(act)
                opp_act = info["opp_action"]
                agent.on_step(act, opp_act)
                ph.append(act); oh.append(opp_act)

                if rnd >= 5 and rnd % 5 == 0:
                    cls_data.append({
                        "features":  extract_features(ph, oh, rnd + 1),
                        "true_type": true_type
                    })
                if done: break
            agent.on_episode_end()

    random.shuffle(cls_data)
    return cls_data

# ─────────────────────────────────────────────────────────────────────────────
# OPPONENT POOLS
# ─────────────────────────────────────────────────────────────────────────────
def build_pools():
    train_specs = [
        (axl.TitForTat,           {}, "TFT-Like"),
        (axl.TitFor2Tats,         {}, "TFT-Like"),
        (axl.SuspiciousTitForTat, {}, "TFT-Like"),
        (axl.HardTitForTat,       {}, "TFT-Like"),
        (axl.SlowTitForTwoTats2,       {}, "TFT-Like"),
        (axl.GTFT,   {}, "TFT-Like"),
        (axl.Grudger,             {}, "Grudger-Like"),
        (axl.Gradual,             {}, "Grudger-Like"),
        (axl.SoftGrudger,         {}, "Grudger-Like"),
        (axl.ForgetfulGrudger,    {}, "Grudger-Like"),
        (axl.Defector,            {}, "Defector-Like"),
        (axl.Aggravater,          {}, "Defector-Like"),
        (axl.TrickyDefector,      {}, "Defector-Like"),
        (axl.HardProber,          {}, "Defector-Like"),
        (axl.Prober,              {}, "Defector-Like"),
        (axl.Random,              {"p": 0.5}, "Stochastic"),
        (axl.Random,              {"p": 0.3}, "Stochastic"),
        (axl.GTFT,                {"p": 0.33}, "Stochastic"),
        (axl.StochasticWSLS,      {"ep": 0.1}, "Stochastic"),
        (axl.Cooperator,          {}, "Cooperative"),
        (axl.Alternator,          {}, "PhaseSwitching"),
        (axl.CyclerCCD,           {}, "PhaseSwitching"),
        (axl.CyclerCCCCCD,        {}, "PhaseSwitching"),
        (axl.CyclerDC,            {}, "PhaseSwitching"),
        (axl.AdaptiveTitForTat,   {}, "Adaptive"),
    ]
    test_specs = [
        (axl.OmegaTFT,             {}, "TFT-Like"),
        (axl.ContriteTitForTat,    {}, "TFT-Like"),
        (axl.EvolvedLookerUp2_2_2, {}, "Grudger-Like"),
        (axl.StochasticWSLS,       {"ep": 0.2}, "Stochastic"),
        (axl.HardProber,           {}, "Defector-Like"),
        (axl.Prober2,              {}, "Defector-Like"),
        (axl.ForgetfulGrudger,     {}, "Grudger-Like"),
        (axl.AdaptiveTitForTat,    {}, "Adaptive"),
        (axl.MetaWinner,           {}, "Sinusoidal"),
        (axl.Alternator,           {}, "PhaseSwitching"),
        (axl.CyclerCCCCCD,         {}, "PhaseSwitching"),
    ]
    def make(specs):
        return [{"class": c, "kwargs": k, "family": f, "name": str(c(**k))}
                for c, k, f in specs]
    return make(train_specs), make(test_specs)

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def train_meta_agent(agent, train_pool, n_rounds=6, seed=SEED):
    print(f"[A-v3-fixed] Training on {len(train_pool)} opponents, {n_rounds} rounds...")
    all_cls = []

    itr = range(n_rounds)
    if _HAS_TQDM: itr = tqdm(itr, desc="Training (A-v3-fixed)")

    for r in itr:
        cls_data = collect_episodes(train_pool, agent, n_eps_per_opp=25, seed=seed+r)
        all_cls.extend(cls_data)

        batch_data = all_cls[-6000:]
        random.shuffle(batch_data)
        losses = []
        for i in range(0, len(batch_data), 512):
            loss = agent.train_classifier(batch_data[i:i+512], n_epochs=20)
            losses.append(loss)

        if _HAS_TQDM:
            sample = random.sample(all_cls, min(400, len(all_cls)))
            feats  = torch.tensor(np.array([e["features"] for e in sample]),
                                  dtype=torch.float32).to(device)
            with torch.no_grad():
                preds = agent.classifier.probs(feats).argmax(-1).cpu().tolist()
            acc = np.mean([p == e["true_type"] for p, e in zip(preds, sample)])
            itr.set_postfix({"loss": f"{np.mean(losses):.4f}", "acc": f"{acc:.3f}"})

    sample = random.sample(all_cls, min(500, len(all_cls)))
    feats  = torch.tensor(np.array([e["features"] for e in sample]),
                          dtype=torch.float32).to(device)
    with torch.no_grad():
        preds = agent.classifier.probs(feats).argmax(-1).cpu().tolist()
    acc = np.mean([p == e["true_type"] for p, e in zip(preds, sample)])
    print(f"[A-v3-fixed] Final classifier accuracy: {acc:.3f}")
    return acc

# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION + ORACLE
# ─────────────────────────────────────────────────────────────────────────────
def evaluate(agent, pool, n_eps=100, seed=SEED):
    results = {}
    for entry in pool:
        env   = IPDEnv(entry["class"], entry["kwargs"], entry["family"])
        oname = f"{entry['family']}/{entry['name']}"
        rewards = []
        for _ in range(n_eps):
            agent.on_episode_start()
            obs = env.reset()
            ep_r = 0.0
            for _ in range(ROUNDS):
                act = agent.select_action(obs, eval_mode=True)
                obs, rew, done, info = env.step(act)
                agent.on_step(act, info["opp_action"])
                ep_r += rew
                if done: break
            agent.on_episode_end()
            rewards.append(ep_r)
        results[oname] = {"avg": float(np.mean(rewards)),
                          "std": float(np.std(rewards)),
                          "family": entry["family"]}
    return results

def compute_oracle(pool, n_eps=50):
    def policy_tft2(ph,oh,r): return D if len(oh)>=2 and oh[-1]==D and oh[-2]==D else C
    policies = [
        ("AllC",  lambda ph,oh,r: C),
        ("AllD",  lambda ph,oh,r: D),
        ("TFT",   policy_tft),
        ("Grim",  policy_grim),
        ("GTFT",  policy_gtft),
        ("TFT2",  policy_tft2),
        ("Cycle", policy_cycler),
    ]
    oracles = {}
    for entry in pool:
        oname = f"{entry['family']}/{entry['name']}"
        best  = 0.0
        for _, pfn in policies:
            total = 0.0
            for _ in range(n_eps):
                env = IPDEnv(entry["class"], entry["kwargs"])
                env.reset()
                ph, oh = [], []
                for rnd in range(ROUNDS):
                    act = pfn(ph, oh, rnd)
                    _, rew, done, info = env.step(act)
                    ph.append(act); oh.append(info["opp_action"])
                    total += rew
                    if done: break
            if total/n_eps > best: best = total/n_eps
        oracles[oname] = best
    return oracles

def print_summary(results, oracle, agent_name):
    print(f"\n{'='*60}")
    print(f"Results: {agent_name}")
    print(f"{'='*60}")
    print(f"{'Opponent':<40} {'Avg R':>8} {'Oracle':>8} {'GS':>7}")
    print(f"{'-'*60}")
    gs_vals = []; by_family = {}
    for oname, r in results.items():
        ora = oracle.get(oname, r["avg"])
        gs  = r["avg"] / max(ora, 1.0)
        gs_vals.append(gs)
        by_family.setdefault(r["family"], []).append(gs)
        print(f"  {oname.split('/')[-1][:38]:<38} {r['avg']:>8.1f} {ora:>8.1f} {gs:>7.3f}")
    print(f"\n  Overall GS: {np.mean(gs_vals):.3f}")
    print(f"  By family:")
    for fam, vals in sorted(by_family.items()):
        print(f"    {fam:<25} {np.mean(vals):.3f}")
    return float(np.mean(gs_vals))

# ─────────────────────────────────────────────────────────────────────────────
# BASELINES
# ─────────────────────────────────────────────────────────────────────────────
class TFTAgent:
    name = "TFT"
    def on_episode_start(self): self._last = C
    def on_episode_end(self): pass
    def select_action(self, obs, eval_mode=False): return self._last
    def on_step(self, a, o, **kw): self._last = o

class GrimAgent:
    name = "Grim"
    def on_episode_start(self): self._d = False
    def on_episode_end(self): pass
    def select_action(self, obs, eval_mode=False): return D if self._d else C
    def on_step(self, a, o, **kw):
        if o == D: self._d = True

class CoopAgent:
    name = "AllC"
    def on_episode_start(self): pass
    def on_episode_end(self): pass
    def select_action(self, obs, eval_mode=False): return C
    def on_step(self, *a, **kw): pass

class DefAgent:
    name = "AllD"
    def on_episode_start(self): pass
    def on_episode_end(self): pass
    def select_action(self, obs, eval_mode=False): return D
    def on_step(self, *a, **kw): pass

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    t0 = time.time()
    os.makedirs("./results_a", exist_ok=True)

    train_pool, test_pool = build_pools()
    print(f"[A-v3-fixed] Train: {len(train_pool)} | Test: {len(test_pool)}")

    print("\n[A-v3-fixed] Computing oracle upper bounds...")
    oracle = compute_oracle(test_pool, n_eps=50)

    agent = MetaAgent(lr_cls=3e-4)
    train_meta_agent(agent, train_pool, n_rounds=6)

    print("\n[A-v3-fixed] Evaluating...")
    res_a = evaluate(agent, test_pool, n_eps=100)
    gs_a  = print_summary(res_a, oracle, "MetaAgent-A-v3-fixed")

    for bl in [TFTAgent(), GrimAgent(), CoopAgent(), DefAgent()]:
        res = evaluate(bl, test_pool, n_eps=100)
        print_summary(res, oracle, bl.name)

    out = {"agent": "MetaAgent-A-v3-fixed", "overall_gs": gs_a,
           "per_opponent": {k: {"avg": v["avg"], "family": v["family"]}
                            for k, v in res_a.items()}}
    with open("./results_a/summary_v3_fixed.json", "w") as f:
        json.dump(out, f, indent=2)

    print(f"\n[A-v3-fixed] Done in {(time.time()-t0)/60:.1f} min")

[A-v3-fixed] Device: cuda
[A-v3-fixed] Train: 25 | Test: 11

[A-v3-fixed] Computing oracle upper bounds...
[A-v3-fixed] Training on 25 opponents, 6 rounds...


Training (A-v3-fixed): 100%|██████████| 6/6 [04:40<00:00, 46.70s/it, loss=1.0236, acc=0.472]


[A-v3-fixed] Final classifier accuracy: 0.494

[A-v3-fixed] Evaluating...

Results: MetaAgent-A-v3-fixed
Opponent                                    Avg R   Oracle      GS
------------------------------------------------------------
  Omega TFT: 3, 8                           300.0    300.0   1.000
  Contrite Tit For Tat                      300.0    300.0   1.000
  EvolvedLookerUp2_2_2                      300.0    312.0   0.962
  Stochastic WSLS: 0.2                      300.0    500.0   0.600
  Hard Prober                               117.5    297.0   0.396
  Prober 2                                  297.0    489.0   0.607
  Forgetful Grudger                         300.0    300.0   1.000
  Adaptive Tit For Tat: 0.5                 300.0    300.0   1.000
  Meta Winner: 220 players                  300.0    500.0   0.600
  Alternator                                293.9    300.0   0.980
  Cycler CCCCCD                             417.7    436.0   0.958

  Overall GS: 0.827
  By fami